# BERT: including misgendering

Fine-tunes and evaluates BERT for the primary outcome.

This notebook accompanies [Stigmatizing Language in Gender-Expansive Patient Records: Corpus Development, Disparity Analysis, and Natural Language Processing-Based Detection Study](https://www.jmir.org/2026/1/e91089).

## Data and execution requirements

- Clinical note text and MIMIC identifiers are not included in this repository.
- Run this notebook only in an environment authorized to access MIMIC-IV and the credentialed annotation release.
- Set `GEP_DATA_DIR`, `GEP_MODEL_DIR`, `GEP_RESULTS_DIR`, and `GEP_FIGURES_DIR` as needed. By default, repository-local directories are used.
- The notebook outputs and execution counters have been removed from the public version.

**Selection protocol.** Epoch selection uses a stratified internal split created only from the official training set. The held-out testing set does not determine an epoch, model, hyperparameter, or decision threshold.


In [ ]:
# Repository-local path configuration
from pathlib import Path
import os

PROJECT_ROOT = Path(os.environ.get("GEP_PROJECT_ROOT", Path.cwd())).resolve()
DATA_DIR = Path(os.environ.get("GEP_DATA_DIR", PROJECT_ROOT / "data")).resolve()
MODEL_DIR = Path(os.environ.get("GEP_MODEL_DIR", PROJECT_ROOT / "models")).resolve()
RESULTS_DIR = Path(os.environ.get("GEP_RESULTS_DIR", PROJECT_ROOT / "results")).resolve()
FIGURES_DIR = Path(os.environ.get("GEP_FIGURES_DIR", PROJECT_ROOT / "figures")).resolve()

for directory in (MODEL_DIR, RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
# ==== Setup & Imports ====
import os
import random
import numpy as np
import pandas as pd
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from threshold_protocol import make_internal_selection_split
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import classification_report, confusion_matrix

# Data and model locations are configured through repository-local paths above.

# ==== Reproducibility ====
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # cudnn deterministic can slow down; enable if needed:
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(42)

# ==== Config ====
MODEL_NAME = "bert-base-uncased"
TRAIN_CSV = str(DATA_DIR / 'GEP_train_80_20.csv')

SAVE_DIR = str(MODEL_DIR / 'BERT_GEP')
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 16
NUM_EPOCHS = 20
LR = 1e-5
DOC_MAX_TOKENS = 4096   # total tokens per document (pre-chunk)
CHUNK_SIZE = 510        # tokens per chunk excluding [CLS]/[SEP]
MAX_LENGTH = 512        # BERT max length (chunk + special tokens)
USE_POS_WEIGHT = False  # set True if your dataset is imbalanced and you want weighting

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ==== Training-only split for epoch selection ====
official_train_df = pd.read_csv(TRAIN_CSV)
train_df, selection_df = make_internal_selection_split(
    official_train_df, label_column="label", selection_fraction=0.15, random_state=42
)
train_texts = train_df["text"].astype(str).tolist()
train_labels = train_df["label"].astype(int).tolist()
val_texts = selection_df["text"].astype(str).tolist()
val_labels = selection_df["label"].astype(int).tolist()
print(f"Internal model-training notes: {len(train_df)}")
print(f"Internal selection notes: {len(selection_df)}")

# ==== Tokenizer & Model ====
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Guard: ensure pad token exists
if tokenizer.pad_token_id is None:
    # Fall back to sep token as pad if necessary
    tokenizer.pad_token = tokenizer.sep_token
    print("No pad_token_id found; using sep_token as pad_token.")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1  # binary with BCEWithLogitsLoss
).to(device)

# ==== Dataset (Chunking & Pooling Support) ====
class ChunkedTextDataset(Dataset):
    """
    Splits long docs into 512-token chunks (with [CLS]/[SEP]),
    caps total tokens per doc at DOC_MAX_TOKENS.
    """
    def __init__(self, texts, labels, tokenizer, chunk_size=510, max_length=512, doc_max_length=4096):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.chunk_size = chunk_size
        self.max_length = max_length
        self.doc_max_length = doc_max_length

    def __len__(self):
        return len(self.texts)

    def chunk_text(self, text):
        # Tokenize without adding special tokens, so we can add CLS/SEP manually
        token_ids = self.tokenizer.encode(text, add_special_tokens=False, truncation=False)
        # Truncate to doc_max_length
        token_ids = token_ids[: self.doc_max_length]

        chunks = []
        for i in range(0, len(token_ids), self.chunk_size):
            core = token_ids[i : i + self.chunk_size]
            chunk = [self.tokenizer.cls_token_id] + core + [self.tokenizer.sep_token_id]
            if len(chunk) < self.max_length:
                chunk = chunk + [self.tokenizer.pad_token_id] * (self.max_length - len(chunk))
            else:
                chunk = chunk[: self.max_length]
            chunks.append(chunk)
        if len(chunks) == 0:
            # Handle empty text: create a single CLS SEP chunk
            chunk = [self.tokenizer.cls_token_id, self.tokenizer.sep_token_id]
            chunk = chunk + [self.tokenizer.pad_token_id] * (self.max_length - len(chunk))
            chunks = [chunk]
        return chunks

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = float(self.labels[idx])  # BCEWithLogitsLoss expects float targets {0.0,1.0}
        chunks = self.chunk_text(text)
        return {
            "chunks": torch.tensor(chunks, dtype=torch.long),  # [num_chunks, max_length]
            "label": torch.tensor(label, dtype=torch.float),   # scalar
            "num_chunks": len(chunks)
        }

def bert_collate_fn(batch):
    all_chunks = [item["chunks"] for item in batch]
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.float)  # [B]
    num_chunks = [item["num_chunks"] for item in batch]
    # Flatten chunks across the batch: [sum_chunks, max_length]
    flat_chunks = torch.cat(all_chunks, dim=0)
    return {"chunks": flat_chunks, "labels": labels, "num_chunks": num_chunks}

train_dataset = ChunkedTextDataset(
    train_texts, train_labels, tokenizer,
    chunk_size=CHUNK_SIZE, max_length=MAX_LENGTH, doc_max_length=DOC_MAX_TOKENS
)
val_dataset = ChunkedTextDataset(
    val_texts, val_labels, tokenizer,
    chunk_size=CHUNK_SIZE, max_length=MAX_LENGTH, doc_max_length=DOC_MAX_TOKENS
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=bert_collate_fn, pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=bert_collate_fn, pin_memory=torch.cuda.is_available()
)

# ==== Optimizer, Loss, Scheduler ====
optimizer = optim.AdamW(model.parameters(), lr=LR)

if USE_POS_WEIGHT:
    # Optional: class weighting for imbalance
    num_neg = (np.array(train_labels) == 0).sum()
    num_pos = (np.array(train_labels) == 1).sum()
    pos_weight_val = (num_neg / max(1, num_pos))
    pos_weight = torch.tensor([pos_weight_val], dtype=torch.float, device=device)
    print(f"Using pos_weight = {pos_weight_val:.4f} (neg={num_neg}, pos={num_pos})")
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
else:
    criterion = nn.BCEWithLogitsLoss()

total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# ==== Train / Eval Loops ====
def train_one_epoch(model, dataloader, criterion, optimizer, device, scheduler):
    model.train()
    total_loss, total, correct = 0.0, 0, 0
    pbar = tqdm(dataloader, desc="Training", ncols=120)
    for batch in pbar:
        chunks = batch["chunks"].to(device)                           # [sum_chunks, max_length]
        labels = batch["labels"].to(device)                           # [B]
        num_chunks = batch["num_chunks"]                               # list of len B

        optimizer.zero_grad()
        outputs = model(input_ids=chunks, attention_mask=(chunks != tokenizer.pad_token_id))
        chunk_logits = outputs.logits.squeeze(-1)                      # [sum_chunks]

        # Pool logits per document (max-logit pooling)
        pooled_logits = []
        idx = 0
        for nc in num_chunks:
            pooled_logits.append(torch.max(chunk_logits[idx: idx + nc]))
            idx += nc
        pooled_logits = torch.stack(pooled_logits)                     # [B]

        loss = criterion(pooled_logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        with torch.no_grad():
            probs = torch.sigmoid(pooled_logits)
            preds = (probs >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            total_loss += loss.item() * labels.size(0)

        pbar.set_postfix({
            "loss": total_loss / max(1, total),
            "acc": 100.0 * correct / max(1, total)
        })

    avg_loss = total_loss / max(1, total)
    avg_acc = 100.0 * correct / max(1, total)
    print(f"[Train] Loss: {avg_loss:.4f} | Acc: {avg_acc:.2f}%")
    return avg_loss, avg_acc

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss, total, correct = 0.0, 0, 0
    all_preds, all_labels = [], []
    pbar = tqdm(dataloader, desc="Validating", ncols=120)
    with torch.no_grad():
        for batch in pbar:
            chunks = batch["chunks"].to(device)
            labels = batch["labels"].to(device)
            num_chunks = batch["num_chunks"]

            outputs = model(input_ids=chunks, attention_mask=(chunks != tokenizer.pad_token_id))
            chunk_logits = outputs.logits.squeeze(-1)

            pooled_logits = []
            idx = 0
            for nc in num_chunks:
                pooled_logits.append(torch.max(chunk_logits[idx: idx + nc]))
                idx += nc
            pooled_logits = torch.stack(pooled_logits)  # [B]

            loss = criterion(pooled_logits, labels)
            total_loss += loss.item() * labels.size(0)

            probs = torch.sigmoid(pooled_logits)
            preds = (probs >= 0.5).float()

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy().astype(int))
            all_labels.extend(labels.cpu().numpy().astype(int))

            pbar.set_postfix({
                "val_loss": total_loss / max(1, total),
                "val_acc": 100.0 * correct / max(1, total)
            })

    avg_loss = total_loss / max(1, total)
    avg_acc = 100.0 * correct / max(1, total)
    print(f"[Valid] Loss: {avg_loss:.4f} | Acc: {avg_acc:.2f}%")
    return avg_loss, avg_acc, np.array(all_preds), np.array(all_labels)

def get_predictions(model, dataloader, device, pooling="max"):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Predicting", ncols=120):
            chunks = batch["chunks"].to(device)
            labels = batch["labels"].to(device)
            num_chunks = batch["num_chunks"]

            outputs = model(input_ids=chunks, attention_mask=(chunks != tokenizer.pad_token_id))
            chunk_logits = outputs.logits.squeeze(-1)

            pooled_logits = []
            idx = 0
            for nc in num_chunks:
                doc_logits = chunk_logits[idx: idx + nc]
                if pooling == "max":
                    pooled = torch.max(doc_logits)
                elif pooling == "mean":
                    pooled = torch.mean(doc_logits)
                else:
                    raise ValueError(f"Unknown pooling: {pooling}")
                pooled_logits.append(pooled)
                idx += nc
            pooled_logits = torch.stack(pooled_logits)

            probs = torch.sigmoid(pooled_logits)
            preds = (probs >= 0.5).long()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy().astype(int))
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_preds), np.array(all_labels), np.array(all_probs)

# ==== Training ====
best_val_acc = 0.0
for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n===== Epoch {epoch}/{NUM_EPOCHS} =====")
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, scheduler)
    print(f"Training  | Loss: {train_loss:.4f} | Acc: {train_acc:.2f}%")

    val_loss, val_acc, val_preds, val_labels_np = evaluate(model, val_loader, criterion, device)
    print(f"Validation| Loss: {val_loss:.4f} | Acc: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        # Save full HF model + tokenizer
        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        print(f" Model saved (new best val acc: {best_val_acc:.2f}%) → {SAVE_DIR}")

        # Report metrics at save time
        cm = confusion_matrix(val_labels_np, val_preds)
        print("Confusion Matrix:")
        print(cm)
        print("Classification Report:")
        print(classification_report(val_labels_np, val_preds, digits=3))

# ==== Final Prediction (optional) ====
# preds, labels_np, probs = get_predictions(model, val_loader, device, pooling="max")
# print("Final Confusion Matrix:")
# print(confusion_matrix(labels_np, preds))
# print("Final Classification Report:")
# print(classification_report(labels_np, preds, digits=3))
